                                                            NBA Player Impact
Goal: In this project we will use advanced/normal basketball stats to quantify what NBA players should be paid. We will use these stats to create a projection of what a player's salary should be. After this, we will subtract the player's projected salary from their actual salary. If that quantiity is positive, we will say that player is underpaid and if it is negative we will say that that player is overpaid. 

In [ ]:
#Pulling the advanced stats from Basketball reference to be our fetures(Sort of a test run)
import pandas as pd

url = "https://www.basketball-reference.com/leagues/NBA_2026_advanced.html"
tables = pd.read_html(url)

advanced = tables[0]
advanced = advanced[advanced["Rk"] != "Rk"]  # remove repeated header rows
print(advanced.head())

    Rk            Player   Age Team Pos     G    GS      MP   PER    TS%  ...  \
0  1.0     Amen Thompson  23.0  HOU  PG  79.0  79.0  2953.0  18.9  0.594  ...   
1  2.0      Kevin Durant  37.0  HOU  SF  78.0  78.0  2840.0  21.1  0.641  ...   
2  3.0      Desmond Bane  27.0  ORL  SG  82.0  82.0  2756.0  16.9  0.607  ...   
3  4.0    Toumani Camara  25.0  POR  PF  82.0  82.0  2731.0  11.3  0.578  ...   
4  5.0  Jabari Smith Jr.  22.0  HOU  PF  77.0  77.0  2705.0  13.5  0.570  ...   

   USG%  OWS  DWS    WS  WS/48  OBPM  DBPM  BPM  VORP          Awards  
0  20.0  6.5  3.8  10.3  0.167   1.6   1.0  2.6   3.4          DPOY-8  
1  27.1  7.5  3.2  10.7  0.180   4.4   0.1  4.5   4.7  CPOY-9,AS,NBA2  
2  23.3  4.8  2.6   7.4  0.129   1.6  -0.2  1.4   2.3             NaN  
3  16.3  2.1  2.7   4.8  0.084  -1.0   0.0 -1.0   0.7             NaN  
4  18.2  3.4  3.3   6.6  0.118  -0.5  -0.2 -0.7   0.9             NaN  

[5 rows x 29 columns]


In [9]:
import unicodedata
import re

# -------------------------
# 1. Helper functions
# -------------------------

def clean_name(name):
    """
    Cleans player names so we can merge across websites.
    Example: Nikola Jokić -> nikola jokic
    """
    name = str(name)
    name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode("utf-8")
    name = name.lower()
    name = re.sub(r"[^a-z\s]", "", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name


def clean_salary(s):
    """
    Converts salary strings like '$55,224,526' into numbers.
    """
    s = str(s)
    s = re.sub(r"[\$,]", "", s)
    return pd.to_numeric(s, errors="coerce")


# -------------------------
# 2. Choose season
# -------------------------

# Basketball Reference uses 2026 for the 2025-26 season
season = 2026

per_game_url = f"https://www.basketball-reference.com/leagues/NBA_{season}_per_game.html"
advanced_url = f"https://www.basketball-reference.com/leagues/NBA_{season}_advanced.html"

# ESPN salary page
salary_url = "https://www.espn.com/nba/salaries"


# -------------------------
# 3. Pull Basketball Reference stats
# -------------------------

per_game = pd.read_html(per_game_url)[0]
advanced = pd.read_html(advanced_url)[0]

# Remove repeated header rows
per_game = per_game[per_game["Rk"] != "Rk"]
advanced = advanced[advanced["Rk"] != "Rk"]

# Keep the columns you want from per-game stats
per_game = per_game[[
    "Player",
    "Age",
    "G",
    "PTS",
    "AST",
    "TRB",
    "BLK"
]]

# Rename columns to your preferred feature names
per_game = per_game.rename(columns={
    "PTS": "PPG",
    "TRB": "REB",
    "BLK": "Blocks",
    "G": "Games_Played"
})

# Keep the columns you want from advanced stats
advanced = advanced[[
    "Player",
    "TS%",
    "WS",
    "USG%",
    "VORP"
]]

# Merge per-game and advanced stats
stats = per_game.merge(advanced, on="Player", how="inner")


# -------------------------
# 4. Fix duplicate players
# -------------------------
# Basketball Reference has multiple rows for traded players.
# The "TOT" row is the full-season total, but since we removed team above,
# duplicates can still happen. This keeps the first row after sorting.

stats["name_clean"] = stats["Player"].apply(clean_name)

# Convert numeric columns
numeric_cols = [
    "Age",
    "Games_Played",
    "PPG",
    "AST",
    "REB",
    "Blocks",
    "TS%",
    "WS",
    "USG%",
    "VORP"
]

for col in numeric_cols:
    stats[col] = pd.to_numeric(stats[col], errors="coerce")

# Drop duplicate player names
stats = stats.drop_duplicates(subset=["name_clean"])


# -------------------------
# 5. Pull salary data
# -------------------------

salary_tables = pd.read_html(salary_url)
salary = salary_tables[0]

# ESPN table came in with numbered columns:
# 0 = rank, 1 = player, 2 = team, 3 = salary
salary = salary.rename(columns={
    1: "Player",
    3: "Salary"
})

# Remove ESPN header row that appears inside the table
salary = salary[salary["Player"] != "NAME"]

# Remove position from ESPN name: "Stephen Curry, G" -> "Stephen Curry"
salary["Player"] = salary["Player"].str.split(",").str[0]

# Clean names for merging
salary["name_clean"] = salary["Player"].apply(clean_name)

# Clean salary numbers
salary["Salary"] = salary["Salary"].apply(clean_salary)
salary["Salary_Millions"] = salary["Salary"] / 1_000_000

salary = salary[["Player", "name_clean", "Salary", "Salary_Millions"]]

print(salary.head())
print(salary.shape)

          Player     name_clean    Salary  Salary_Millions
1  Stephen Curry  stephen curry  59606817        59.606817
2    Joel Embiid    joel embiid  55224526        55.224526
3   Nikola Jokic   nikola jokic  55224526        55.224526
4   Kevin Durant   kevin durant  54708609        54.708609
5  Anthony Davis  anthony davis  54126450        54.126450
(40, 4)


In [10]:
# -------------------------
# 6. Clean salary table
# -------------------------

# Adjust these column names if your printed columns look different
salary = salary.rename(columns={
    "NAME": "Player",
    "SALARY": "Salary"
})

salary["name_clean"] = salary["Player"].apply(clean_name)
salary["Salary"] = salary["Salary"].apply(clean_salary)
salary["Salary_Millions"] = salary["Salary"] / 1_000_000

salary = salary[["Player", "name_clean", "Salary", "Salary_Millions"]]


# -------------------------
# 7. Merge stats with salary
# -------------------------

df = stats.merge(
    salary,
    on="name_clean",
    how="inner",
    suffixes=("", "_salary")
)

# Final columns for your project
df = df[[
    "Player",
    "PPG",
    "AST",
    "REB",
    "TS%",
    "WS",
    "USG%",
    "Games_Played",
    "Age",
    "Blocks",
    "VORP",
    "Salary_Millions"
]]

print(df.head())
print(df.shape)

                    Player   PPG  AST  REB    TS%    WS  USG%  Games_Played  \
0              Luka Dončić  33.5  8.3  7.7  0.616   9.5  38.1          64.0   
1  Shai Gilgeous-Alexander  31.1  6.6  4.3  0.665  15.2  33.4          68.0   
2          Anthony Edwards  28.8  3.7  5.0  0.617   6.5  31.5          61.0   
3             Jaylen Brown  28.7  5.1  6.9  0.573   6.9  36.2          71.0   
4             Tyrese Maxey  28.3  6.6  4.1  0.588   8.7  29.4          70.0   

    Age  Blocks  VORP  Salary_Millions  
0  26.0     0.5   6.6        54.126450  
1  27.0     0.8   7.8        38.333050  
2  24.0     0.8   3.5        45.550512  
3  29.0     0.4   3.3        53.142264  
4  25.0     0.8   4.9        37.958760  
(38, 12)


In [11]:
#setting up our fetures and labels for our linear regression(In this case we want a linear regression because we aren't predicting a yes or no question)
features = [
    "PPG",
    "AST",
    "REB",
    "TS%",
    "WS",
    "USG%",
    "Games_Played",
    "Age",
    "Blocks",
    "VORP"
]

label = "Salary_Millions"

X = df[features]
y = df[label]

#True shooting and Usage come in as strings so convert them into numbers
for col in features + [label]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=features + [label])



In [ ]:
#Now do the test train split(80% training and 20% test data)
from sklearn.model_selection import train_test_split
#Do the linear regression after
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R^2:", r2)

MAE: 4.55100279529129
RMSE: 6.153649299759065
R^2: 0.16672380237035467


In [ ]:
#Now we see who is overpaid and who is underpaid
results = X_test.copy()
results["Player"] = df.loc[X_test.index, "Player"]
results["Actual_Salary"] = y_test
results["Predicted_Salary"] = y_pred
results["Salary_Difference"] = results["Predicted_Salary"] - results["Actual_Salary"]

results = results[[
    "Player",
    "Actual_Salary",
    "Predicted_Salary",
    "Salary_Difference"
]]

print(results.sort_values("Salary_Difference", ascending=False))#prints 8 players because our test set is 20% of the data

                Player  Actual_Salary  Predicted_Salary  Salary_Difference
4         Tyrese Maxey      37.958760         45.913948           7.955188
30      Darius Garland      39.446090         40.652242           1.206152
6     Donovan Mitchell      46.394100         45.770341          -0.623759
33      Scottie Barnes      38.661750         37.563995          -1.097755
36          OG Anunoby      39.568966         36.033574          -3.535392
13        Kevin Durant      54.708609         50.729850          -3.978759
26         LaMelo Ball      37.958760         33.753113          -4.205647
27  Karl-Anthony Towns      54.126450         40.321080         -13.805370
